# 08 · Delta Lake et Iceberg

La table des courses devient « vivante » : on ajoute la deuxième quinzaine de mars, on supprime des courses erronées et on garde l'historique de chaque version.
On utilise Delta Lake en Python (bibliothèque `deltalake`, sans Spark), puis Iceberg en bonus.

## 1. Pourquoi un format de table ?

Un simple dossier de fichiers Parquet n'a ni transaction (un job qui plante laisse des fichiers à moitié écrits), ni UPDATE ni DELETE, ni historique.
Delta Lake et Iceberg ajoutent un **journal** au-dessus des fichiers Parquet : il dit quels fichiers forment la table, à chaque version.

In [1]:
from pathlib import Path
import json
import shutil
import pandas as pd
from deltalake import DeltaTable, write_deltalake

DATA = Path("../data")
OUTPUT = Path("../output")
courses = pd.read_parquet(DATA / "yellow_tripdata_2026-03.parquet")

On coupe le mois en deux : les courses parties avant le 16 mars, puis les autres.

In [2]:
courses_1_15 = courses[courses["tpep_pickup_datetime"] < "2026-03-16"]
courses_16_31 = courses[courses["tpep_pickup_datetime"] >= "2026-03-16"]
len(courses_1_15), len(courses_16_31)

(1960351, 1992100)

1 960 351 et 1 992 100 courses : environ 2 millions dans chaque moitié.

## 2. Version 0 : créer la table

`write_deltalake` écrit un DataFrame en table Delta. `shutil.rmtree` efface d'abord l'ancien dossier, pour pouvoir relancer le notebook.

In [3]:
lac = OUTPUT / "delta" / "courses"
shutil.rmtree(lac, ignore_errors=True)
write_deltalake(lac, courses_1_15)
DeltaTable(lac).version()

0

La table existe : c'est la version 0.

## 3. Regarder dans le dossier

Une petite fonction affiche les fichiers d'un dossier avec leur taille.

In [4]:
def lister(dossier):
    """Affiche chaque fichier du dossier (et des sous-dossiers) avec sa taille en Ko."""
    for fichier in sorted(dossier.rglob("*")):
        if fichier.is_file():
            print(fichier.relative_to(dossier), "-", fichier.stat().st_size // 1000, "Ko")

lister(lac)

_delta_log/00000000000000000000.json - 4 Ko
part-00000-91caab1b-7559-4350-9594-43dac8e37f83-c000.snappy.parquet - 48726 Ko


Les données sont un fichier Parquet ordinaire (environ 49 Mo).
Le sous-dossier `_delta_log` contient le **journal** : un petit fichier JSON par version.

In [5]:
journal = lac / "_delta_log" / "00000000000000000000.json"
for ligne in journal.read_text().splitlines():
    print(list(json.loads(ligne).keys()))

['commitInfo']
['protocol']
['metaData']
['add']


Chaque ligne du journal est une action : `commitInfo` (quand, quelle opération), `protocol` (version du format), `metaData` (le schéma) et `add` (le fichier Parquet ajouté).
Tant que ce fichier JSON n'est pas écrit, la version n'existe pas pour les lecteurs : c'est la **transaction**.

## 4. Version 1 : ajouter des données

`mode="append"` ajoute la deuxième quinzaine ; `count()` donne le nombre de lignes de la table.

In [6]:
write_deltalake(lac, courses_16_31, mode="append")
dt = DeltaTable(lac)
dt.version(), dt.count()

(1, 3952451)

Version 1 : la table contient tout le mois, soit 3 952 451 courses.

## 5. Version 2 : supprimer des lignes

Une course de 0 mile est sans doute une erreur. `delete` supprime les lignes qui vérifient une condition écrite comme en SQL.

In [7]:
dt.delete("trip_distance = 0")

{'num_added_files': 1,
 'num_removed_files': 2,
 'num_deleted_rows': 121210,
 'num_copied_rows': 3831241,
 'execution_time_ms': 2362,
 'scan_time_ms': 23,
 'rewrite_time_ms': 0}

121 210 courses supprimées (`num_deleted_rows`), 3 831 241 conservées (`num_copied_rows`).
Delta ne modifie pas un fichier Parquet : il en écrit un nouveau sans ces lignes (`num_added_files`) et retire les 2 anciens de la table (`num_removed_files`).

## 6. L'historique

`history()` relit le journal : une ligne par version (l'heure, en millisecondes, est convertie en date).

In [8]:
historique = pd.DataFrame(dt.history())
historique["timestamp"] = pd.to_datetime(historique["timestamp"], unit="ms")
historique[["version", "timestamp", "operation", "operationParameters"]]

,version,timestamp,operation,operationParameters
0,2,2026-10-09 22:56:54.358,DELETE,{'predicate': 'trip_distance = 0'}
1,1,2026-10-09 22:56:51.979,WRITE,{'mode': 'Append'}
2,0,2026-10-09 22:56:51.138,WRITE,{'mode': 'ErrorIfExists'}


Trois versions, la plus récente en premier : suppression, ajout (`Append`) et création (`ErrorIfExists`, le mode par défaut : erreur si la table existe déjà).
Chacune garde son heure (UTC) et ses paramètres : c'est utile pour un audit.

## 7. Remonter le temps (*time travel*)

On relit la table telle qu'elle était en version 1, avant la suppression (seulement la colonne `trip_distance`, pour ménager la mémoire).

In [9]:
avant = DeltaTable(lac, version=1).to_pandas(columns=["trip_distance"])
print(len(avant), "lignes, dont", (avant["trip_distance"] == 0).sum(), "à distance nulle")

3952451 lignes, dont 121210 à distance nulle


Les 121 210 courses supprimées sont toujours là dans la version 1 : les anciens fichiers Parquet n'ont pas été effacés.

## 8. Faire le ménage : `vacuum`

`vacuum` liste les fichiers qui ne servent plus à la dernière version ; par défaut il n'efface rien (`dry_run=True`) et garde 7 jours d'historique.
Pour la démonstration, on met la rétention à 0 heure, ce qui oblige à lever une sécurité (`enforce_retention_duration=False`).

In [10]:
dt.vacuum(retention_hours=0, enforce_retention_duration=False)

['part-00000-35938273-9d94-4331-a038-d8f1cd975b86-c000.snappy.parquet',
 'part-00000-91caab1b-7559-4350-9594-43dac8e37f83-c000.snappy.parquet']

Les 2 fichiers retirés par le `delete` (ceux des versions 0 et 1) seraient effacés.
Avec `dry_run=False`, on gagne de la place, mais le *time travel* vers ces versions devient impossible.

## 9. Bonus : la même idée avec Iceberg

Iceberg a besoin d'un **catalogue** qui retrouve les tables : ici une petite base SQLite. On y crée un espace de noms `taxis` (un groupe de tables).
Les chemins doivent être absolus : `resolve()`, puis `as_uri()` pour l'adresse `file:///…`.

In [11]:
import pyarrow as pa
from pyiceberg.catalog.sql import SqlCatalog

entrepot = (OUTPUT / "iceberg").resolve()
shutil.rmtree(entrepot, ignore_errors=True)
entrepot.mkdir(parents=True)
catalogue = SqlCatalog("ama", uri=f"sqlite:///{entrepot}/catalog.db",
                       warehouse=entrepot.as_uri())
catalogue.create_namespace("taxis")
catalogue.list_namespaces()

[('taxis',)]

On crée la table `taxis.courses` avec la première quinzaine, puis on lit seulement les courses payées en espèces (`payment_type == 2`).

In [12]:
table = pa.Table.from_pandas(courses_1_15, preserve_index=False)
courses_iceberg = catalogue.create_table("taxis.courses", schema=table.schema)
courses_iceberg.append(table)
especes = courses_iceberg.scan(row_filter="payment_type == 2").to_pandas()
len(especes)

167755

167 755 courses en espèces sur la première quinzaine.
Le filtre est appliqué pendant la lecture, comme les `filters` de Parquet.

In [13]:
lister(entrepot)

catalog.db - 20 Ko
taxis/courses/data/00000-0-a95af3ff-b02e-4d3a-afce-2f24d04fed5f.parquet - 34115 Ko
taxis/courses/metadata/00000-1ce94c4d-2117-4de9-9177-29a917068804.metadata.json - 1 Ko
taxis/courses/metadata/00001-094c9109-3ffa-450c-b13c-9605c1f74eb4.metadata.json - 2 Ko
taxis/courses/metadata/a95af3ff-b02e-4d3a-afce-2f24d04fed5f-m0.avro - 5 Ko
taxis/courses/metadata/snap-5103280951363771604-0-a95af3ff-b02e-4d3a-afce-2f24d04fed5f.avro - 1 Ko


On retrouve la chaîne du cours : `catalog.db` → `metadata.json` → liste de *manifests* (`snap-….avro`) → *manifest* (`…-m0.avro`) → fichiers Parquet de `data/`.
Chaque écriture ajoute un `metadata.json` : `00000` à la création, `00001` après l'ajout.

## Exercices

### ✍️ Exercice 1 — Supprimer les montants négatifs

Supprimez de la table Delta les courses dont `total_amount` est négatif, affichez l'historique, puis comptez les montants négatifs dans la version précédente (version 2).

In [14]:
# ✅ Solution
dt.delete("total_amount < 0")
historique = pd.DataFrame(dt.history())
print(historique[["version", "operation", "operationParameters"]])
precedente = DeltaTable(lac, version=2).to_pandas(columns=["total_amount"])
print((precedente["total_amount"] < 0).sum(), "montants négatifs dans la version 2")

   version operation                 operationParameters
0        3    DELETE   {'predicate': 'total_amount < 0'}
1        2    DELETE  {'predicate': 'trip_distance = 0'}
2        1     WRITE                  {'mode': 'Append'}
3        0     WRITE           {'mode': 'ErrorIfExists'}
19283 montants négatifs dans la version 2


La version 3 a supprimé les montants négatifs (sans doute des remboursements ou des annulations).
La version 2 contient toujours ces 19 283 courses : on peut encore la relire.

### ✍️ Exercice 2 — Où sont passés les fichiers ?

Comptez les fichiers Parquet présents dans le dossier Delta, puis ceux qu'utilisent la version 1 (avant le premier `delete`) et la version actuelle (`file_uris()`).
Pourquoi les fichiers « supprimés » sont-ils encore sur le disque ?

In [15]:
# ✅ Solution
print("sur le disque :", len(list(lac.glob("*.parquet"))))
print("version 1 :", len(DeltaTable(lac, version=1).file_uris()))
print("version actuelle :", len(dt.file_uris()))

sur le disque : 4
version 1 : 2
version actuelle : 1


4 fichiers Parquet sur le disque, 2 utilisés par la version 1, un seul par la version actuelle.
Un `delete` n'efface aucun fichier : le journal marque seulement les anciens comme retirés (`remove`) ; ils restent pour le *time travel* jusqu'à un `vacuum` avec `dry_run=False`.

## À retenir

- Delta Lake et Iceberg = des fichiers Parquet + un journal qui dit quels fichiers forment la table.
- Chaque écriture crée une nouvelle version : ajout (`append`), suppression (`delete`), avec un historique.
- *Time travel* : on relit une ancienne version tant que ses fichiers n'ont pas été effacés par `vacuum`.
- Delta n'a besoin que d'un dossier ; Iceberg a besoin d'un catalogue.
- Pour un simple échange de fichiers, Parquet ou CSV suffisent ; le format de table sert aux tables vivantes.